<H1 style="font-family:verdana"> <img src="assets/lbd_base.drawio.svg" style="width: 60px;" alt="[LBD logo]" />
    How to run your PySpark jobs on the LBD cluster</H1>

**Author:** Giovanna Roda

When you run a PySpark job—like those found in most online tutorials—it will, by default, execute on your local machine (e.g., your virtual machine or pod).

# PySpark “Hello World”

This is the classic **“Hello World”** of PySpark — the moment where you bring Spark to life.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Spark test") \
    .getOrCreate()

sc = spark.sparkContext

print(f"Context: {sc}")
# your code

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 12:42:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Context: <SparkContext master=local[*] appName=Spark test>


## Breaking it down

* **`SparkSession`**
  This is your **entry point into Spark**. Think of it as the control center that lets you interact with the Spark engine.

* **`.appName("Spark test")`**
  This gives your application a name.
  It’s not just cosmetic — this name shows up in the Spark UI and logs, which is very helpful when multiple jobs are running.

* **`.getOrCreate()`**
  This is Spark being polite:

  * If a session already exists → reuse it
  * Otherwise → create a new one

* **`spark.sparkContext`**
  The **SparkContext** is the lower-level engine underneath.
  It’s responsible for actually connecting to the cluster (or your local machine) and executing tasks.

* **`spark.stop()`**
  Clean shutdown. Always a good habit — like turning off the lights when you leave.

> With just a few lines of code, you’re not just running Python anymore — you’re starting a distributed data processing engine.

## But where did this job run?

If you look closely at the output you'll see:
```
Context: <SparkContext master=local[*] appName=Spark test>
```

This line reveals something important 👀

* **`master=local[*]`**
  This means Spark is running in **local mode** — on your own machine (e.g. virtual machine or pod).
  The `[*]` tells Spark to use **all available CPU cores**.

* **`appName=Spark test`**
  That’s the name we assigned earlier. It shows up here, confirming that this is *your* application.


### What does this mean?

Even though you're using **Spark**, nothing is actually distributed here.
Your job is running entirely inside your **VM or pod**, just like a regular Python program — only using multiple threads.

### What does this *really* mean?

Your job **is parallelized**, but only within a **single machine**:

* ✅ Multiple tasks run **in parallel (multi-threaded)**
* ✅ All available vCPUs/cores can be used
* ❌ No distribution across multiple nodes
* ❌ No cluster resource management involved


> So yes — Spark *is* doing parallel work here.
> But it’s confined to a single machine: your VM or pod.
>
> Think of it as “mini-Spark” — powerful, but not yet distributed.

# The real power of Spark

> The real power of Spark comes when the same code runs not just across cores… but across machines.

Whether a Spark runs on your laptop or a cluster of machines depends entirely on how your session is configured.

> So the real question is:
> how do we get from `local[*]`… to an actual cluster?

## Can we control where Spark runs?

The short answer is: **Yes, you can set both programmatically**, but there are some important technical gotchas regarding when and how they take effect.

## 1. Setting the Master

You can easily define the **execution environment** (the *Master*) directly in your Python script:

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyAppData") \
    .master("local[*]") \
    .getOrCreate()
```

* **`.master("local[*]")`** → run locally using all CPU cores
* Other options include:
  `local`, `local[K]`, `spark://host:port`, `mesos://host:port`, or `yarn`

💡 In our example earlier, we didn’t set this explicitly — so Spark quietly defaulted to `local[*]`.


## 2. Setting the Deploy Mode

This is where things get interesting.

You *can* try to set:

```python
.config("spark.submit.deployMode", "cluster")
```

…but in practice, this **does not behave the way you might expect**.


### Why?

The **Deploy Mode** decides *where the Driver runs*:

* **Client mode** → Driver runs where you launched the script (e.g., your VM/pod)
* **Cluster mode** → Driver runs inside the cluster (on a worker node)

Here’s the catch:

> Once your Python script has started, the decision is already made.

Trying to switch to cluster mode *inside* the script is like:

> trying to move a running train onto a different track 🚂

You’re already in **client mode**.


## 3. What works where?

| Method                    | Set Master | Set Deploy Mode | When to use                    |
| ------------------------- | ---------- | --------------- | ------------------------------ |
| **Inside Python script**  | ✅ Yes      | ❌ Not really    | Local dev, quick tests         |
| **`spark-submit`**        | ✅ Yes      | ✅ Yes           | **Production / real clusters** |
| **`spark-defaults.conf`** | ✅ Yes      | ✅ Yes           | Cluster-wide defaults          |

---

## 4. Demo takeaway 💡

Right now, in your demo:

* You are running with
  `master=local[*]` → **parallelized on one machine**
* You are implicitly in
  **client mode** → Driver runs in your VM/pod


> So the real upgrade is not changing the code…
>
> It’s changing *where it runs*.



### Quick checklist

* ✅ **Local / learning / demos** → use `.master("local[*]")`
* 🚀 **Real cluster (YARN, Kubernetes)** → use `spark-submit`
* ⚠️ **Notebooks (Jupyter, Zeppelin)** → almost always **client mode**


# Same code, real cluster

## Running Spark from a Pod with `spark-submit`

With the current LBD pod setup, you **cannot fully switch to cluster mode from inside Python**.
You need to:

* **`--master yarn`** → run the job on the cluster
* **`--deploy-mode cluster`** → run the Driver on a cluster node

Save your script (e.g., `spark_test.py`) and run it in a `%%bash` cell:

```bash
%%bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  spark_test.py
```

**Note:**

* In **cluster mode**, the Driver runs on a cluster node, **not your pod**
* Logs will appear on the node that hosted the Driver, not locally

In [2]:
%%bash
cat >$HOME/spark_test.py <<🐱
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Spark test") \
    .getOrCreate()

sc = spark.sparkContext

print(f"Context: {sc}")
# your code

spark.stop()
🐱

Overwriting /home/smoketest/spark_test.py


In [3]:
%%bash
export HADOOP_CONF_DIR=/etc/hadoop/conf
export SPARK_HOME=/usr/lib/spark
export JAVA_HOME=$(ls -d /usr/lib/jvm/*-17*-openjdk* 2>/dev/null | head -n1)

spark-submit \
  --master yarn \
  --deploy-mode cluster \
  $HOME/spark_test.py

26/04/02 12:42:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 12:42:39 INFO AHSProxy: Connecting to Application History server at lbdmg01.datalab.novalocal/10.11.12.189:10200
26/04/02 12:42:40 INFO Configuration: resource-types.xml not found
26/04/02 12:42:40 INFO ResourceUtils: Unable to find 'resource-types.xml'.
26/04/02 12:42:40 INFO Client: Verifying our application has not requested more than the maximum memory capability of the cluster (8192 MB per container)
26/04/02 12:42:40 INFO Client: Will allocate AM container, with 1408 MB memory including 384 MB overhead
26/04/02 12:42:40 INFO Client: Setting up container launch context for our AM
26/04/02 12:42:40 INFO Client: Setting up the launch environment for our AM container
26/04/02 12:42:40 INFO Client: Preparing resources for our AM container
26/04/02 12:42:40 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back

# Conclusion

PySpark **inside a notebook** cannot run in **cluster deploy-mode**.
The only way to run the Driver on a cluster node is to:

1. Save your code to a `.py` file
2. Run it with **`spark-submit`** from a terminal or `%%bash` cell

> Once submitted in cluster mode, your job runs independently — you can close your pod or JupyterHub, and nothing is lost. Logs and results remain on the cluster.


See also: [basicSpark_Pi_EstimationLocalVsCluster.ipynb](basicSpark_Pi_EstimationLocalVsCluster.ipynb)